# P2 — Sparse-event GEE + T1 serialization audit (CPU and GPU)

**Two independent outputs:** `P2_CPU_SPARSE_GEE_T1_FORENSICS.zip` and `P2_GPU_T1_STANDARDIZED_ROBUSTNESS.zip`.

**Protocol status:** exploratory robustness checks, **not a replacement** for the original `results.rar` experiments or the separate E1/E2 paired-baseline analysis.

**Runtime:** Choose **T4 GPU** (or A100) to run both parts by **Runtime → Run all**. CPU analysis still runs on the CPU, and the two ZIPs are independent. A CPU-only runtime performs CPU work but creates an explicit GPU `NOT_RUN` report instead of fabricating predictions.

**CPU input (automatic):** the eleven frozen `paired_cases_part_*.csv` from the public repository. **Optional T1 input:** upload a text-bearing CSV/ZIP with columns `case_id`, `prompt_id`, `prompt`, `output`, and optionally `gold_label`, `authorized_context`. GPU evaluation requires **all 720 unique T1 cases**. `prompt` is the original question/task and `output` is the generator output being evaluated; do not substitute gold labels for text.

**GPU:** Qwen3-8B and Mistral-7B-Instruct-v0.3 are loaded sequentially in 4-bit mode to fit a 15+ GiB Colab GPU when feasible. Results use a *new standardized prompt* and forced choice A/B, not historical bitwise reproduction. Qwen thinking is disabled in the chat template where supported. Memory allocation can fail on free-tier cards; failure is recorded.


In [ ]:
!pip -q install pandas numpy scipy scikit-learn statsmodels huggingface_hub transformers accelerate bitsandbytes sentencepiece tqdm

In [ ]:
from pathlib import Path
import os, json, sys, hashlib, zipfile, time, traceback, gc, subprocess, shutil
import numpy as np, pandas as pd
from datetime import datetime, timezone
from IPython.display import display
ROOT=Path('/content/P2_DUAL_AUDIT'); ROOT.mkdir(exist_ok=True)
CPU=ROOT/'CPU'; GPU=ROOT/'GPU'; INPUT=ROOT/'INPUT'
for p in [CPU,GPU,INPUT]:p.mkdir(parents=True,exist_ok=True)
SEED=20261015; np.random.seed(SEED)
REPO='fatmas1982/semantic-coherence-recovery'; BASE=f'https://raw.githubusercontent.com/{REPO}/main/'
def save_json(path,obj):Path(path).write_text(json.dumps(obj,indent=2,default=str),encoding='utf-8')
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def package(folder,out):
    with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
        for f in sorted(folder.rglob('*')):
            if f.is_file():z.write(f,f.relative_to(folder))
    print('CREATED',out,Path(out).stat().st_size,'bytes')
def fetch(url,target):
    import requests
    r=requests.get(url,timeout=90);r.raise_for_status();Path(target).write_bytes(r.content)
    return sha(target)
manifest={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'repository':REPO,'seed':SEED,'python':sys.version,'scope':'new diagnostics only; RAR and E1/E2 estimates unchanged'}


## CPU 1 — Download and validate original frozen paired data (automatic)

In [ ]:
input_hashes={}
for i in range(1,12):
    name=f'paired_cases_part_{i:02d}.csv'; path=INPUT/name
    input_hashes[name]=fetch(BASE+'data/validity_2026_10/'+name,path)
df=pd.concat([pd.read_csv(INPUT/f'paired_cases_part_{i:02d}.csv',dtype={'case_id':str,'prompt_id':str}) for i in range(1,12)],ignore_index=True)
KEYS=['judge','target','experiment','variant']
assert len(df)==10408 and df.groupby(KEYS).ngroups==20,(len(df),df.groupby(KEYS).ngroups)
assert not df.duplicated(KEYS+['case_id']).any()
assert df.groupby(KEYS).prompt_id.nunique().min()==120
assert (df.pred_base.ne(df.pred_pert).astype(int)==df.changed.astype(int)).all()
print('PASS: 10408 paired observations, 20 comparisons, no duplicate cases.')
save_json(CPU/'INPUT_SHA256.json',input_hashes)


## CPU 2 — Exact frozen GEE specification and sparse-event inspection

Re-fit GEE with robust Prompt_ID-clustered SEs, GOLD class and anonymous generator slot adjustment. Diagnose the three low-event *contrasts* by number of events, exposed/unexposed events, standard errors and OR intervals. **30 is a warning threshold, not a universal sample-size rule.**

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
AXES={'primary':'primary_disagree','boundary':'boundary_disagree','severity':'severity_disagree'}
rows=[]
for key,g in df.groupby(KEYS,sort=True):
    for axis,column in AXES.items():
        z=g.copy();z['D']=z[column].astype(int)
        X=pd.get_dummies(z[['D','gold_label','generator_id']].astype({'gold_label':'category','generator_id':'category'}),columns=['gold_label','generator_id'],drop_first=True,dtype=float)
        X=sm.add_constant(X.astype(float))
        y=z.changed.astype(float)
        try:
            fit=sm.GEE(y,X,groups=z.prompt_id,family=sm.families.Binomial(),cov_struct=sm.cov_struct.Independence()).fit(maxiter=150)
            beta=float(fit.params['D']);se=float(fit.bse['D']);p=float(fit.pvalues['D']); status='PASS' if fit.converged else 'NONCONVERGED'
            OR=float(np.exp(beta));low=float(np.exp(beta-1.96*se));high=float(np.exp(beta+1.96*se))
        except Exception as exc:
            beta=se=p=OR=low=high=np.nan;status=f'FAILED: {str(exc)[:150]}'
        rec=dict(zip(KEYS,key));rec.update(human_axis=axis,n_cases=len(z),prompt_clusters=z.prompt_id.nunique(),total_changed=int(y.sum()),events_disagree=int(((y==1)&(z.D==1)).sum()),events_agree=int(((y==1)&(z.D==0)).sum()),n_disagree=int(z.D.sum()),n_agree=int((z.D==0).sum()),num_parameters=X.shape[1],events_per_parameter=float(y.sum()/X.shape[1]),gee_adjusted_OR=OR,OR_Wald95_low=low,OR_Wald95_high=high,robust_SE=se,gee_p=p,convergence=status,low_event_flag=bool(y.sum()<30))
        rows.append(rec)
gee=pd.DataFrame(rows)
assert len(gee)==60
gee['holm_p']=np.nan;gee['bh_q']=np.nan
ok=np.isfinite(gee.gee_p)
if ok.all():
    gee['holm_p']=multipletests(gee.gee_p,method='holm')[1]
    gee['bh_q']=multipletests(gee.gee_p,method='fdr_bh')[1]
gee.to_csv(CPU/'GEE_60_RECOMPUTED_DIAGNOSTICS.csv',index=False)
sparse=gee[gee.low_event_flag].copy();sparse.to_csv(CPU/'GEE_SPARSE_EVENT_DIAGNOSTICS.csv',index=False)
print('GEE fits',len(gee),'low-event',len(sparse),'Holm discoveries',int((gee.holm_p<.05).sum()));display(sparse)
if len(sparse)!=3:print('WARNING: expected 3 low-event fits from previous freeze; inspect provenance!')


## CPU 3 — Robustness of interpretation and cluster influence

Compare full 60-test conclusion with the subset of adequately populated tests **without re-adjusting Holm** (re-adjustment would change the hypothesis family). For each sparse contrast, leave out each Prompt_ID cluster once and re-fit; track convergence and the range of the OR. This is influence diagnosis, **not a new confirmatory test**.

In [ ]:
influence=[]
for _,r in sparse.iterrows():
    mask=np.ones(len(df),dtype=bool)
    for k in KEYS:mask &= df[k].eq(r[k]).to_numpy()
    z0=df.loc[mask].copy();column=AXES[r.human_axis]
    for pid in z0.prompt_id.unique():
        z=z0.loc[z0.prompt_id!=pid].copy();z['D']=z[column].astype(int)
        X=pd.get_dummies(z[['D','gold_label','generator_id']].astype({'gold_label':'category','generator_id':'category'}),columns=['gold_label','generator_id'],drop_first=True,dtype=float)
        X=sm.add_constant(X.astype(float))
        try:
            fit=sm.GEE(z.changed.astype(float),X,groups=z.prompt_id,family=sm.families.Binomial(),cov_struct=sm.cov_struct.Independence()).fit(maxiter=100)
            estimate=float(np.exp(fit.params['D']));status='PASS' if fit.converged else 'NONCONVERGED'
        except Exception as e:estimate=np.nan;status='FAILED:'+str(e)[:70]
        rec={k:r[k] for k in KEYS};rec.update(human_axis=r.human_axis,omitted_prompt_id=pid,OR=estimate,status=status)
        influence.append(rec)
inf=pd.DataFrame(influence);inf.to_csv(CPU/'GEE_SPARSE_LEAVE_ONE_PROMPT_OUT.csv',index=False)
full_discoveries=gee.loc[gee.holm_p<.05,KEYS+['human_axis']].to_dict('records')
non_sparse=gee[~gee.low_event_flag]
conclusion={'n_fits':60,'sparse_fits':int(len(sparse)),'full_family_holm_discoveries':full_discoveries,'full_discovery_count':len(full_discoveries),'sparse_discovery_count':int((sparse.holm_p<.05).sum()),'non_sparse_discovery_count_at_original_holm':int((non_sparse.holm_p<.05).sum()),'reduced_set_p_values_recorrected':False,'leave_one_cluster_fit_failures':int((inf.status!='PASS').sum()),'note':'Sparse observations retained in original results. This is an exploratory influence audit, not external validation.'}
save_json(CPU/'SPARSE_GEE_ROBUSTNESS_CONCLUSION.json',conclusion)
print(json.dumps(conclusion,indent=2))


## CPU 4 — Optional T1 prompt-text forensic audit

Upload an authentic source CSV/ZIP of **the 720 T1 cases**, optionally with old serialized strings `serialized_fp16_reference` and `serialized_e1_rot0`. If old serialized strings are absent, the notebook reports `NOT_RECONSTRUCTIBLE` rather than claiming that it explains historical hash differences.

If your input names differ, rename its columns to `case_id`, `prompt_id`, `prompt`, `output` (and optional `gold_label`, `authorized_context`).

In [ ]:
# Change to False to skip interactive upload; GPU part will be marked NOT_RUN if no valid data.
UPLOAD_T1=True
T1_PATH=None
if UPLOAD_T1:
    from google.colab import files
    print('OPTIONAL BUT REQUIRED FOR GPU: choose a T1 cases CSV or ZIP containing CSV. Cancel upload if unavailable.')
    up=files.upload()
    for filename,content in up.items():
        dest=INPUT/Path(filename).name;dest.write_bytes(content)
        if dest.suffix.lower()=='.csv':T1_PATH=dest
        elif dest.suffix.lower()=='.zip':
            with zipfile.ZipFile(dest) as archive:
                candidates=[p for p in archive.namelist() if p.lower().endswith('.csv') and not p.startswith('__MACOSX/')]
                if len(candidates)==1:
                    T1_PATH=INPUT/'T1_FROM_ZIP.csv';T1_PATH.write_bytes(archive.read(candidates[0]))
                else:print('ZIP has',len(candidates),'CSVs. Upload one consolidated T1 CSV instead.')
T1_OK=False;T1=None
if T1_PATH:
    T1=pd.read_csv(T1_PATH,dtype={'case_id':str,'prompt_id':str})
    req={'case_id','prompt_id','prompt','output'}
    problems=[]
    if not req.issubset(T1.columns):problems.append('MISSING_COLUMNS '+str(sorted(req-set(T1.columns))))
    if len(T1)!=720:problems.append(f'EXPECTED_720_GOT_{len(T1)}')
    if 'case_id' in T1 and T1.case_id.duplicated().any():problems.append('DUPLICATE_CASE_ID')
    if {'prompt','output'}.issubset(T1.columns) and (T1[['prompt','output']].isna().any().any()):problems.append('EMPTY_PROMPT_OR_OUTPUT')
    T1_OK=not problems
    save_json(CPU/'T1_INPUT_VALIDATION.json',{'valid':T1_OK,'n_rows':len(T1),'columns':list(T1.columns),'problems':problems,'sha256':sha(T1_PATH)})
else:
    save_json(CPU/'T1_INPUT_VALIDATION.json',{'valid':False,'reason':'No original text-bearing T1 input supplied. Case-level paired E1/E2 CSVs do not contain the generator text.'})
if T1_OK:
    hashes=[]
    for _,r in T1.iterrows():
        z={'case_id':r.case_id,'prompt_id':r.prompt_id,'prompt_sha256':hashlib.sha256(str(r['prompt']).encode()).hexdigest(),'output_sha256':hashlib.sha256(str(r['output']).encode()).hexdigest()}
        for col in ['serialized_fp16_reference','serialized_e1_rot0']:
            if col in T1.columns:z[col+'_SHA256']=hashlib.sha256(str(r[col]).encode()).hexdigest()
        if {'serialized_fp16_reference','serialized_e1_rot0'}.issubset(T1.columns):z['serialized_texts_equal']=str(r['serialized_fp16_reference'])==str(r['serialized_e1_rot0'])
        hashes.append(z)
    pd.DataFrame(hashes).to_csv(CPU/'T1_AUTHENTIC_TEXT_HASHES.csv',index=False)
    if 'serialized_texts_equal' in hashes[0]:
        status='HISTORICAL_SERIALIZED_TEXT_AVAILABLE';nmatch=int(sum(x['serialized_texts_equal'] for x in hashes))
    else:status='ORIGINAL_SERIALIZED_PROMPTS_NOT_PROVIDED';nmatch=None
    save_json(CPU/'T1_FORENSIC_FINDING.json',{'status':status,'identical_serializations':nmatch,'n_cases':len(T1),'limitation':'Reconstructed standardized prompts are not original FP16 reference invocations.'})
else:save_json(CPU/'T1_FORENSIC_FINDING.json',{'status':'NOT_RECONSTRUCTIBLE','limitation':'Need original T1 generator outputs and prompt strings.'})
print('T1 original case text available:',T1_OK)


## CPU 5 — Finalize and download independent CPU ZIP

In [ ]:
manifest.update(input_sha256=input_hashes,cpu_results={'gee_models':len(gee),'low_event_models':len(sparse),'leave_one_cluster_runs':len(inf),'T1_text_valid':T1_OK})
save_json(CPU/'CPU_RUN_MANIFEST.json',manifest)
(CPU/'README.txt').write_text('CPU-only: 60 GEE fits, sparse-event diagnostics, leave-one-prompt-cluster-out influence and optional authentic T1 text checks. Does not modify historical RAR or sensitivity estimates.\n',encoding='utf-8')
CPU_ZIP=Path('/content/P2_CPU_SPARSE_GEE_T1_FORENSICS.zip');package(CPU,CPU_ZIP)
from google.colab import files
files.download(str(CPU_ZIP))


## GPU 1 — Standardized independent T1 robustness (requires T1 text + CUDA)

Uses independently specified standardized T1 binary detection task. **Not historical FP16 reference / E1 rot0 reproduction.** Both judges see the same instructions and case text after model-specific template conversion. Each model is loaded and unloaded sequentially. Output records per-case token probabilities/scores, prompt hashes and failures, with status indicated.


In [ ]:
import torch
GPU_AVAILABLE=bool(torch.cuda.is_available())
GPU_STATUS={'cuda_available':GPU_AVAILABLE,'t1_valid':T1_OK,'experiment':'new standardized T1 robustness; not source-run replacement','models':{}}
print('GPU',torch.cuda.get_device_name(0) if GPU_AVAILABLE else 'none','T1_OK',T1_OK)


In [ ]:
# Conservative settings: test on 12 cases by default before increasing to 720.
# For the FULL intended experiment, set MAX_T1_CASES = 720.
MAX_T1_CASES=720
MODELS={'qwen3_8b':'Qwen/Qwen3-8B','mistral_7b':'mistralai/Mistral-7B-Instruct-v0.3'}
MAX_INPUT_TOKENS=4096
BATCH_LOG_EVERY=20
RUBRIC=('You are scoring semantic failure in an LLM output. Classify whether the output exhibits a substantive semantic-coherence failure relative to the user request and any authorized context. '
        'Answer with one label only: A = no substantive failure, B = substantive failure. No explanation. '
        'Do not use any gold labels; use only the case prompt, source context if provided, and model output.')
def message_for(row):
    ctx=str(row.get('authorized_context','')) if pd.notna(row.get('authorized_context','')) else ''
    return f'Case prompt:\n{row["prompt"]}\n\nAuthorized context:\n{ctx}\n\nGenerated response:\n{row["output"]}\n\nSelect A or B.'
def score_model(alias,model_id,data):
    from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
    tok=AutoTokenizer.from_pretrained(model_id,trust_remote_code=False)
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16)
    model=AutoModelForCausalLM.from_pretrained(model_id,quantization_config=quant,device_map='auto',torch_dtype=torch.float16,trust_remote_code=False)
    model.eval()
    device=next(model.parameters()).device
    ids={}
    for c in ['A','B']:
        variants=[tok.encode(c,add_special_tokens=False),tok.encode(' '+c,add_special_tokens=False)]
        ids[c]=variants[0]
        if len(ids[c])!=1:raise ValueError(f'{alias}: answer label {c} is not a single token in context-independent tokenizer; STOP')
    if ids['A'][0]==ids['B'][0]:raise ValueError('Answer-token collision')
    results=[]
    for ix,(_,row) in enumerate(data.iterrows()):
        messages=[{'role':'system','content':RUBRIC},{'role':'user','content':message_for(row)}]
        try:
            kwargs={'tokenize':False,'add_generation_prompt':True}
            if alias.startswith('qwen'):kwargs['enable_thinking']=False
            try:formatted=tok.apply_chat_template(messages,**kwargs)
            except TypeError:kwargs.pop('enable_thinking',None);formatted=tok.apply_chat_template(messages,**kwargs)
            encoding=tok(formatted,return_tensors='pt',truncation=True,max_length=MAX_INPUT_TOKENS)
            token_length=int(encoding['input_ids'].shape[1]);enc={k:v.to(device) for k,v in encoding.items()}
            with torch.inference_mode(): logits=model(**enc).logits[0,-1,:].float()
            pair=torch.stack([logits[ids['A'][0]],logits[ids['B'][0]]]);probs=torch.softmax(pair,dim=0).detach().cpu().numpy()
            rec={'case_id':str(row.case_id),'prompt_id':str(row.prompt_id),'judge':alias,'status':'ok','decision':'A' if probs[0]>=probs[1] else 'B','prob_A_conditional':float(probs[0]),'prob_B_conditional':float(probs[1]),'token_count':token_length,'input_sha256':hashlib.sha256(formatted.encode('utf-8')).hexdigest(),'serialized_prompt_preview':formatted[:160]}
            if 'gold_label' in data.columns:rec['gold_label']=row.get('gold_label')
        except Exception as e:
            rec={'case_id':str(row.case_id),'prompt_id':str(row.prompt_id),'judge':alias,'status':'failed','error':str(e)[:240]}
        results.append(rec)
        if (ix+1)%BATCH_LOG_EVERY==0:
            pd.DataFrame(results).to_csv(GPU/f'{alias}_T1_PARTIAL.csv',index=False)
            print(alias,ix+1,'/',len(data),'success',sum(r['status']=='ok' for r in results))
    pd.DataFrame(results).to_csv(GPU/f'{alias}_T1_STANDARDIZED_PREDICTIONS.csv',index=False)
    del model,tok;gc.collect();torch.cuda.empty_cache()
    return results
if not GPU_AVAILABLE or not T1_OK:
    GPU_STATUS['status']='NOT_RUN_MISSING_GPU_OR_VALID_T1_TEXT'
else:
    data=T1.iloc[:min(MAX_T1_CASES,len(T1))].copy()
    for alias,model_id in MODELS.items():
        try:
            prediction=score_model(alias,model_id,data)
            n_ok=sum(p.get('status')=='ok' for p in prediction)
            GPU_STATUS['models'][alias]={'expected':len(data),'ok':n_ok,'failed':len(data)-n_ok,'model_id':model_id}
        except Exception as e:
            GPU_STATUS['models'][alias]={'status':'FAILED','error':str(e)[:800]}
            (GPU/f'{alias}_ERROR.txt').write_text(traceback.format_exc(),encoding='utf-8')
        gc.collect();torch.cuda.empty_cache()
    GPU_STATUS['status']='COMPLETED' if len(GPU_STATUS['models'])==2 and all(x.get('ok')==len(data) for x in GPU_STATUS['models'].values()) else 'PARTIAL_OR_FAILED'
print(GPU_STATUS)


## GPU 2 — Pair models and save honest outcomes; download second ZIP

A successful full experiment requires two complete 720-row model prediction CSVs. If some calls fail or text was not supplied, the ZIP contains a clear `NOT_RUN`/`PARTIAL` status and never claims scientific completion. This new comparison cannot explain historic T1 reference hash differences unless original serialized prompts are available.


In [ ]:
try:
    a=GPU/'qwen3_8b_T1_STANDARDIZED_PREDICTIONS.csv';b=GPU/'mistral_7b_T1_STANDARDIZED_PREDICTIONS.csv'
    if a.exists() and b.exists():
        x=pd.read_csv(a);y=pd.read_csv(b)
        pairs=x.merge(y,on='case_id',suffixes=('_qwen','_mistral'),validate='one_to_one')
        pairs['judge_disagree']=pairs.decision_qwen.ne(pairs.decision_mistral)
        pairs.to_csv(GPU/'T1_CROSS_JUDGE_PAIRED_COMPARISON.csv',index=False)
        GPU_STATUS['paired_n']=len(pairs)
        GPU_STATUS['cross_judge_disagreement_rate']=float(pairs.judge_disagree.mean())
        GPU_STATUS['caveat']='different architectures/templates; not isolating architecture or adjudicator accuracy'
except Exception as e:GPU_STATUS['pairing_error']=str(e)
if T1_OK:GPU_STATUS['t1_file_sha256']=sha(T1_PATH)
save_json(GPU/'GPU_RUN_MANIFEST.json',GPU_STATUS)
(GPU/'README.txt').write_text('This is a NEW standardized T1 robustness run, not exact reconstruction of historical reference/rot0 prompting. Read GPU_RUN_MANIFEST.json and verify 720 ok cases per judge before interpreting.\n',encoding='utf-8')
GPU_ZIP=Path('/content/P2_GPU_T1_STANDARDIZED_ROBUSTNESS.zip');package(GPU,GPU_ZIP)
from google.colab import files
files.download(str(GPU_ZIP))
print('FINISHED. Two independent ZIP files: CPU and GPU. Review manifests for PASS, PARTIAL, or NOT_RUN.')
